<a href="https://colab.research.google.com/github/Thanjaivalavan/M2_GenAI_AgenticAI/blob/main/03_gan_toy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Slides 67-70: Generative Adversarial Network (GAN) — minimax objective
-------------------------------------------------------------------------
Implements, from scratch (tiny numpy MLPs, no PyTorch/TF)

    min_G max_D V(D,G) = E_x~p_data[log D(x)] + E_z~p_z[log(1 - D(G(z)))]

    Discriminator wants to MAXIMIZE V:
        - push D(x)    -> 1   for real data
        - push D(G(z)) -> 0   for fake data

    Generator wants to MINIMIZE V (equivalently maximize log D(G(z))
    to avoid the vanishing-gradient version of the original loss):
        - push D(G(z)) -> 1   (fool the discriminator)

    At convergence (Nash equilibrium): D*(x) = 0.5 everywhere.

Toy task: learn to generate 1-D samples from a target Gaussian
N(mean=4, std=0.5) starting from random noise z ~ N(0,1).
"""

import numpy as np

np.random.seed(0)


# ---------------------------------------------------------------------------
# Tiny 2-layer MLP (numpy only) shared by both G and D
# ---------------------------------------------------------------------------
class MLP:
    def __init__(self, in_dim, hidden_dim, out_dim, out_activation):
        self.W1 = np.random.randn(in_dim, hidden_dim) * 0.5
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(hidden_dim, out_dim) * 0.5
        self.b2 = np.zeros(out_dim)
        self.out_activation = out_activation

    def forward(self, x):
        self.x = x
        self.z1 = x @ self.W1 + self.b1
        self.a1 = np.tanh(self.z1)                       # hidden activation
        self.z2 = self.a1 @ self.W2 + self.b2
        if self.out_activation == "sigmoid":
            self.out = 1 / (1 + np.exp(-self.z2))         # D(x) in (0,1)
        else:
            self.out = self.z2                            # G(z): raw value
        return self.out

    def backward(self, d_out, lr):
        # d_out: gradient of loss wrt self.out
        if self.out_activation == "sigmoid":
            d_z2 = d_out * self.out * (1 - self.out)       # sigmoid'
        else:
            d_z2 = d_out

        d_W2 = self.a1.T @ d_z2
        d_b2 = d_z2.sum(axis=0)
        d_a1 = d_z2 @ self.W2.T
        d_z1 = d_a1 * (1 - self.a1 ** 2)                   # tanh'
        d_W1 = self.x.T @ d_z1
        d_b1 = d_z1.sum(axis=0)
        d_x = d_z1 @ self.W1.T                             # gradient wrt input (needed by G)

        self.W2 -= lr * d_W2
        self.b2 -= lr * d_b2
        self.W1 -= lr * d_W1
        self.b1 -= lr * d_b1
        return d_x


def sample_real(n):
    """p_data: target distribution we want the Generator to learn."""
    return np.random.normal(loc=4.0, scale=0.5, size=(n, 1))


def sample_noise(n):
    """p_z: the Generator's input noise distribution."""
    return np.random.normal(loc=0.0, scale=1.0, size=(n, 1))


def train_gan(epochs=3000, batch_size=64, lr=0.01, eps=1e-8):
    G = MLP(in_dim=1, hidden_dim=16, out_dim=1, out_activation="linear")
    D = MLP(in_dim=1, hidden_dim=16, out_dim=1, out_activation="sigmoid")

    for epoch in range(epochs):
        # ---- 1. Train Discriminator: MAXIMIZE E[log D(x)] + E[log(1-D(G(z)))] ----
        x_real = sample_real(batch_size)
        z = sample_noise(batch_size)
        x_fake = G.forward(z)

        d_real = D.forward(x_real)
        d_fake = D.forward(x_fake)

        # Discriminator loss = -(mean log D(x_real) + mean log(1 - D(x_fake)))
        d_loss = -np.mean(np.log(d_real + eps) + np.log(1 - d_fake + eps))

        # dLoss/d(d_real) = -1/d_real ; dLoss/d(d_fake) = 1/(1-d_fake)
        grad_real = (-1.0 / (d_real + eps)) / batch_size
        D.forward(x_real)
        D.backward(grad_real, lr)

        grad_fake = (1.0 / (1 - d_fake + eps)) / batch_size
        D.forward(x_fake)
        D.backward(grad_fake, lr)

        # ---- 2. Train Generator: MINIMIZE log(1-D(G(z)))  ==  MAXIMIZE log D(G(z)) ----
        z = sample_noise(batch_size)
        x_fake = G.forward(z)
        d_fake = D.forward(x_fake)

        g_loss = -np.mean(np.log(d_fake + eps))   # non-saturating generator loss
        grad_d_fake = (-1.0 / (d_fake + eps)) / batch_size
        d_x_fake = D.backward(grad_d_fake, lr=0.0)  # get gradient wrt D's input, don't update D here
        G.backward(d_x_fake, lr)

        if epoch % 500 == 0 or epoch == epochs - 1:
            print(f"epoch {epoch:4d} | D_loss={d_loss:.4f}  G_loss={g_loss:.4f}  "
                  f"D(real) avg={d_real.mean():.3f}  D(fake) avg={d_fake.mean():.3f}")

    return G, D


def main():
    print("Target real distribution: N(mean=4.0, std=0.5)")
    print("Training GAN so G(z) learns to imitate it...\n")
    G, D = train_gan()

    z = sample_noise(500)
    generated = G.forward(z)
    print(f"\nAfter training, G(z) samples: mean={generated.mean():.3f}, std={generated.std():.3f}")
    print("(should be close to the target mean=4.0, std=0.5)")

    x_real = sample_real(500)
    d_on_real = D.forward(x_real).mean()
    d_on_fake = D.forward(generated).mean()
    print(f"\nD(real) avg = {d_on_real:.3f}   D(fake) avg = {d_on_fake:.3f}")
    print("At a perfect Nash equilibrium both would be ~0.5 (D can no longer tell them apart).")


if __name__ == "__main__":
    main()


Target real distribution: N(mean=4.0, std=0.5)
Training GAN so G(z) learns to imitate it...

epoch    0 | D_loss=1.2856  G_loss=0.6983  D(real) avg=0.553  D(fake) avg=0.497
epoch  500 | D_loss=1.3969  G_loss=0.7082  D(real) avg=0.488  D(fake) avg=0.493
epoch 1000 | D_loss=1.3985  G_loss=0.6835  D(real) avg=0.498  D(fake) avg=0.505
epoch 1500 | D_loss=1.3833  G_loss=0.6944  D(real) avg=0.501  D(fake) avg=0.499
epoch 2000 | D_loss=1.3829  G_loss=0.6982  D(real) avg=0.499  D(fake) avg=0.497
epoch 2500 | D_loss=1.3883  G_loss=0.6899  D(real) avg=0.501  D(fake) avg=0.502
epoch 2999 | D_loss=1.3877  G_loss=0.6926  D(real) avg=0.499  D(fake) avg=0.500

After training, G(z) samples: mean=4.067, std=0.321
(should be close to the target mean=4.0, std=0.5)

D(real) avg = 0.499   D(fake) avg = 0.500
At a perfect Nash equilibrium both would be ~0.5 (D can no longer tell them apart).
